In [0]:
import pyspark.sql.functions as F

In [0]:
df = spark.read.table("workspace.bronze.erp_customer")
display(df.printSchema())
df.show()

In [0]:
# trim() string fields
for col_name, dtype in df.dtypes:
    if dtype == 'string':
        df = df.withColumn(col_name, F.trim(F.col(col_name)))
df.printSchema()

In [0]:
# cleanup gender
df = df.withColumn(
    'GEN',
    F.when(F.col('GEN') == 'M', 'Male')
    .when(F.col('GEN') == 'F', 'Female')
    .when(F.col('GEN') == 'NULL', 'N/A')
    .when(F.col('GEN') == '', 'N/A')
    .when(F.col("GEN").isNull(), 'N/A')
    .otherwise(F.col('GEN'))
)
df.select("GEN").distinct().show()

In [0]:
# cleanup ID
df = df.withColumn(
    'CID',
    F.when(F.col("CID").startswith("NAS"), F.col("CID").substr(4, 10))
    .otherwise(F.col("CID"))
)
df.show()

In [0]:
# rename
mapping_names = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "GEN": "gender"
}
for old_name, new_name in mapping_names.items():
    df = df.withColumnRenamed(old_name, new_name)
df.printSchema()

In [0]:
df.show()

In [0]:
# write 
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_customer")